# ESCO Skill Linking Diagnostics

This notebook audits the ESCO skill linking step used by the KG reranking experiments.

It reproduces the paper configuration:
- ESCO encoder: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`
- query linking: top-20 ESCO skills
- document linking: top-15 ESCO skills
- minimum cosine similarity: `0.45`

The goal is to produce the missing robustness analysis requested by reviewers:
- coverage and zero-link rate
- average number of retained links
- similarity-score distribution
- a compact paper-ready table in CSV and LaTeX format
- a few qualitative examples and likely error cases

Note: precision/recall for skill linking cannot be measured directly here because the repo does not contain gold query-skill or item-skill annotations. This notebook therefore focuses on coverage, retained-link counts, score distributions, and qualitative error analysis.

In [1]:
import os
import re
import unicodedata
from pathlib import Path
from typing import List, Optional

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

PROJECT_ROOT = Path("/Users/user/Projects/question-retrieval-KG").resolve()
QB_PATH = Path("/Users/user/question-retrieval-KIPerWeb/testbeds/open-datafold.csv")
QUERIES_PATH = Path("/Users/user/question-retrieval-KIPerWeb/testbeds/queries_experiments/queries/queries.csv")
SKILLS_PATH = Path("/Users/user/Downloads/ESCO/skills_de.csv")

OUT_DIR = PROJECT_ROOT / "artifacts" / "linking_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ESCO_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ESCO_TOPK_DOC = 15
ESCO_TOPK_QUERY = 20
ESCO_MIN_SIM = 0.45

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)

/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/user/Projects/question-retrieval-KG
OUT_DIR: /Users/user/Projects/question-retrieval-KG/artifacts/linking_analysis


In [2]:
_ws_re = re.compile(r"\\s+")
_tags_re = re.compile(r"<[^>]+>")
_tok_re = re.compile(r"[a-zA-ZäöüÄÖÜß]+")

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _tags_re.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    s = re.sub(r"[^0-9a-zA-ZäöüßÄÖÜẞ]+", " ", s)
    s = _ws_re.sub(" ", s).strip()
    return s

def safe_concat(parts: List[Optional[str]]) -> str:
    parts = [p for p in parts if p is not None]
    clean = []
    for p in parts:
        p = str(p)
        if p.strip() in ("", "N/A", "nan"):
            continue
        clean.append(p)
    return " ".join(clean)

def build_doc_text(row: pd.Series) -> str:
    return safe_concat([row.get("question"), row.get("choices_processed")])

qb = pd.read_csv(QB_PATH).fillna("N/A")
queries_file = pd.read_csv(QUERIES_PATH).fillna("")

qb["docno"] = qb["test_item_id"].astype(str)
qb["raw_text"] = qb.apply(build_doc_text, axis=1)
qb["text"] = qb["raw_text"].map(normalize_text)

topics = queries_file.rename(columns={"queries": "query"})[["qid", "query"]].copy()
topics["qid"] = topics["qid"].astype(str)
topics["raw_query"] = topics["query"].astype(str)
topics["query"] = topics["query"].astype(str).map(normalize_text)

corpus = qb[["docno", "raw_text", "text"]].copy()

print("corpus:", corpus.shape)
print("topics:", topics.shape)
corpus.head(2)

corpus: (2812, 3)
topics: (15, 3)


,docno,raw_text,text
0,2812,Was ist die Grundlage der Elektrotechnik? Mech...,was ist die grundlage der elektrotechnik mecha...
1,2813,Was ist eine elektrische Schaltung? Ein Comput...,was ist eine elektrische schaltung ein compute...


In [3]:
skills_df = pd.read_csv(SKILLS_PATH).fillna("")
skills_df["conceptUri"] = skills_df["conceptUri"].astype(str).str.strip()
skills_df["preferredLabel"] = skills_df["preferredLabel"].astype(str).str.strip()

skills_nn = skills_df[["conceptUri", "preferredLabel"]].drop_duplicates("conceptUri").copy()
skills_nn = skills_nn[(skills_nn["conceptUri"] != "") & (skills_nn["conceptUri"] != "nan")]
skills_nn = skills_nn[(skills_nn["preferredLabel"] != "") & (skills_nn["preferredLabel"] != "nan")].reset_index(drop=True)

skill_vocab = skills_nn["conceptUri"].tolist()
skill_labels = skills_nn["preferredLabel"].tolist()

st_esco = SentenceTransformer(ESCO_MODEL)
X = st_esco.encode(
    skill_labels,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index_esco = faiss.IndexFlatIP(X.shape[1])
index_esco.add(X)

print("skills indexed:", len(skill_vocab), "embedding dim:", X.shape[1])

def top_skills(texts: List[str], topk: int):
    Q = st_esco.encode(
        [str(t) for t in texts],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")
    scores, idxs = index_esco.search(Q, int(topk))
    rows = []
    for row_id, (doc_scores, doc_idxs) in enumerate(zip(scores, idxs)):
        for j, sc in zip(doc_idxs.tolist(), doc_scores.tolist()):
            if j < 0:
                continue
            if float(sc) < float(ESCO_MIN_SIM):
                continue
            rows.append((row_id, skill_vocab[j], skill_labels[j], float(sc)))
    return rows

Batches:   0%|          | 0/436 [00:00<?, ?it/s]

Batches:   0%|          | 1/436 [00:00<01:51,  3.92it/s]

Batches:   1%|          | 3/436 [00:00<00:59,  7.31it/s]

Batches:   1%|          | 5/436 [00:00<00:47,  9.10it/s]

Batches:   2%|▏         | 8/436 [00:00<00:33, 12.96it/s]

...

Batches:  98%|█████████▊| 427/436 [00:15<00:00, 40.69it/s]

Batches:  99%|█████████▉| 432/436 [00:15<00:00, 38.63it/s]

Batches: 100%|██████████| 436/436 [00:15<00:00, 36.20it/s]

Batches: 100%|██████████| 436/436 [00:15<00:00, 28.08it/s]

skills indexed: 13939 embedding dim: 768


In [4]:
doc_pairs = top_skills(corpus["text"].astype(str).tolist(), topk=ESCO_TOPK_DOC)
qid_pairs = top_skills(topics["query"].astype(str).tolist(), topk=ESCO_TOPK_QUERY)

doc_links = pd.DataFrame(doc_pairs, columns=["row_id", "skill_uri", "skill_label", "sim"])\
    .merge(corpus.reset_index().rename(columns={"index": "row_id"}), on="row_id", how="left")

query_links = pd.DataFrame(qid_pairs, columns=["row_id", "skill_uri", "skill_label", "sim"])\
    .merge(topics.reset_index().rename(columns={"index": "row_id"}), on="row_id", how="left")

doc_counts = doc_links.groupby("docno").size().reindex(corpus["docno"], fill_value=0)
query_counts = query_links.groupby("qid").size().reindex(topics["qid"], fill_value=0)

doc_links.to_csv(OUT_DIR / "doc_skill_links.csv", index=False)
query_links.to_csv(OUT_DIR / "query_skill_links.csv", index=False)
doc_counts.rename("n_links").to_csv(OUT_DIR / "doc_link_counts.csv", index=True)
query_counts.rename("n_links").to_csv(OUT_DIR / "query_link_counts.csv", index=True)

print("saved raw linking artifacts to", OUT_DIR)

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   1%|          | 1/88 [00:00<00:28,  3.08it/s]

Batches:   2%|▏         | 2/88 [00:00<00:26,  3.29it/s]

Batches:   3%|▎         | 3/88 [00:00<00:25,  3.40it/s]

Batches:   5%|▍         | 4/88 [00:01<00:24,  3.43it/s]

...

Batches: 100%|██████████| 88/88 [00:16<00:00, 10.30it/s]

Batches: 100%|██████████| 88/88 [00:16<00:00,  5.37it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.01it/s]

saved raw linking artifacts to /Users/user/Projects/question-retrieval-KG/artifacts/linking_analysis


In [5]:
def summarize_links(unit_name: str, counts: pd.Series, link_df: pd.DataFrame, topk: int) -> dict:
    sims = link_df["sim"].astype(float)
    return {
        "Unit": unit_name,
        "N": int(len(counts)),
        "Top-k": int(topk),
        "Min sim.": float(ESCO_MIN_SIM),
        "Coverage (%)": float((counts > 0).mean() * 100.0),
        "Zero-link n": int((counts == 0).sum()),
        "Full top-k (%)": float((counts == topk).mean() * 100.0),
        "Avg. kept links": float(counts.mean()),
        "Median kept links": float(counts.median()),
        "Mean sim.": float(sims.mean()) if len(sims) else np.nan,
        "Median sim.": float(sims.median()) if len(sims) else np.nan,
        "p10 sim.": float(sims.quantile(0.10)) if len(sims) else np.nan,
        "p90 sim.": float(sims.quantile(0.90)) if len(sims) else np.nan,
    }

summary_full = pd.DataFrame([
    summarize_links("Query", query_counts, query_links, ESCO_TOPK_QUERY),
    summarize_links("Document", doc_counts, doc_links, ESCO_TOPK_DOC),
])

summary_paper = summary_full[[
    "Unit", "Top-k", "Min sim.", "Coverage (%)", "Zero-link n",
    "Full top-k (%)", "Avg. kept links", "Median sim.", "p10 sim.", "p90 sim."
]].copy()

summary_paper["Coverage (%)"] = summary_paper["Coverage (%)"].map(lambda x: f"{x:.1f}")
summary_paper["Min sim."] = summary_paper["Min sim."].map(lambda x: f"{x:.2f}")
summary_paper["Full top-k (%)"] = summary_paper["Full top-k (%)"].map(lambda x: f"{x:.1f}")
summary_paper["Avg. kept links"] = summary_paper["Avg. kept links"].map(lambda x: f"{x:.1f}")
summary_paper["Median sim."] = summary_paper["Median sim."].map(lambda x: f"{x:.3f}")
summary_paper["p10 sim."] = summary_paper["p10 sim."].map(lambda x: f"{x:.3f}")
summary_paper["p90 sim."] = summary_paper["p90 sim."].map(lambda x: f"{x:.3f}")

summary_full.to_csv(OUT_DIR / "linking_stats_full.csv", index=False)
summary_paper.to_csv(OUT_DIR / "linking_stats_table_paper.csv", index=False)

latex_table = summary_paper.to_latex(index=False, escape=False, column_format="lrrrrrrrrr")
(OUT_DIR / "linking_stats_table_paper.tex").write_text(latex_table, encoding="utf-8")

summary_paper

,Unit,Top-k,Min sim.,Coverage (%),Zero-link n,Full top-k (%),Avg. kept links,Median sim.,p10 sim.,p90 sim.
0,Query,20,0.45,100.0,0,100.0,20.0,0.713,0.642,0.815
1,Document,15,0.45,99.6,12,94.2,14.6,0.608,0.517,0.713


## Paper-ready table

The table below is the compact version intended for the paper. It is also exported as:
- `artifacts/linking_analysis/linking_stats_table_paper.csv`
- `artifacts/linking_analysis/linking_stats_table_paper.tex`

In [6]:
summary_paper

,Unit,Top-k,Min sim.,Coverage (%),Zero-link n,Full top-k (%),Avg. kept links,Median sim.,p10 sim.,p90 sim.
0,Query,20,0.45,100.0,0,100.0,20.0,0.713,0.642,0.815
1,Document,15,0.45,99.6,12,94.2,14.6,0.608,0.517,0.713


In [7]:
doc_row = summary_full.loc[summary_full["Unit"] == "Document"].iloc[0]
query_row = summary_full.loc[summary_full["Unit"] == "Query"].iloc[0]

paragraph = (
    f"Under the paper configuration (top-{ESCO_TOPK_QUERY} query links, top-{ESCO_TOPK_DOC} document links, cosine >= {ESCO_MIN_SIM:.2f}), "
    f"ESCO linking covers {query_row['Coverage (%)']:.1f}% of queries and {doc_row['Coverage (%)']:.1f}% of documents. "
    f"Queries retain {query_row['Avg. kept links']:.1f} links on average, while documents retain {doc_row['Avg. kept links']:.1f}; "
    f"only {int(query_row['Zero-link n'])} queries and {int(doc_row['Zero-link n'])} documents fall back to zero vectors. "
    f"Similarity scores are higher for queries (median {query_row['Median sim.']:.3f}, p10-p90 {query_row['p10 sim.']:.3f}-{query_row['p90 sim.']:.3f}) "
    f"than for documents (median {doc_row['Median sim.']:.3f}, p10-p90 {doc_row['p10 sim.']:.3f}-{doc_row['p90 sim.']:.3f}). "
    f"The threshold filters almost none of the query links ({query_row['Full top-k (%)']:.1f}% still keep the full top-{ESCO_TOPK_QUERY}), "
    f"and only modestly affects documents ({doc_row['Full top-k (%)']:.1f}% still keep the full top-{ESCO_TOPK_DOC})."
)

print(paragraph)
(OUT_DIR / "linking_summary_paragraph.txt").write_text(paragraph, encoding="utf-8")

Under the paper configuration (top-20 query links, top-15 document links, cosine >= 0.45), ESCO linking covers 100.0% of queries and 99.6% of documents. Queries retain 20.0 links on average, while documents retain 14.6; only 0 queries and 12 documents fall back to zero vectors. Similarity scores are higher for queries (median 0.713, p10-p90 0.642-0.815) than for documents (median 0.608, p10-p90 0.517-0.713). The threshold filters almost none of the query links (100.0% still keep the full top-20), and only modestly affects documents (94.2% still keep the full top-15).


573

## Qualitative examples

These examples are meant to support a short robustness discussion. They are exported for easier reuse in the paper draft or rebuttal.

In [8]:
query_top3 = query_links.sort_values(["qid", "sim"], ascending=[True, False]).groupby("qid").head(3).copy()
query_top3["rank"] = query_top3.groupby("qid").cumcount() + 1
query_examples = query_top3[["qid", "raw_query", "rank", "skill_label", "sim"]].copy()
query_examples.to_csv(OUT_DIR / "query_examples_top3.csv", index=False)

zero_link_docs = corpus.loc[doc_counts.eq(0).values, ["docno", "raw_text"]].copy()
zero_link_docs["raw_text"] = zero_link_docs["raw_text"].astype(str).str.slice(0, 240)
zero_link_docs.to_csv(OUT_DIR / "zero_link_documents.csv", index=False)

stop = {"und","oder","der","die","das","ein","eine","im","in","mit","von","für","des","dem","den","am","an","auf","zu"}
def content_tokens(s: str):
    return {t.lower() for t in _tok_re.findall(str(s)) if len(t) > 3 and t.lower() not in stop}

top1 = query_links.sort_values(["qid", "sim"], ascending=[True, False]).groupby("qid").head(1).copy()
top1["query_tokens"] = top1["raw_query"].map(content_tokens)
top1["skill_tokens"] = top1["skill_label"].map(content_tokens)
top1["token_overlap_n"] = top1.apply(lambda r: len(r["query_tokens"] & r["skill_tokens"]), axis=1)
top1["overlap_tokens"] = top1.apply(lambda r: ", ".join(sorted(r["query_tokens"] & r["skill_tokens"])), axis=1)
suspicious_queries = top1.sort_values(["token_overlap_n", "sim"], ascending=[True, True])[[
    "qid", "raw_query", "skill_label", "sim", "token_overlap_n", "overlap_tokens"
]].copy()
suspicious_queries.to_csv(OUT_DIR / "query_examples_potential_errors.csv", index=False)

display(query_examples.head(12))
display(zero_link_docs.head(10))
display(suspicious_queries.head(10))

,qid,raw_query,rank,skill_label,sim
0,0,Digitaler Schutz,1,digitale Sicherheitsmaßnahmen anwenden,0.904953
1,0,Digitaler Schutz,2,IKT-Geräte schützen,0.822421
2,0,Digitaler Schutz,3,IKT-Sicherheit,0.756137
20,1,Einfache Datenanalyse,1,Datenanalyse,0.931369
21,1,Einfache Datenanalyse,2,Datenanalyse durchführen,0.917756
22,1,Einfache Datenanalyse,3,Daten untersuchen,0.834657
200,10,Damen im Berufsleben,1,die Gleichstellung der Geschlechter am Arbeits...,0.676048
201,10,Damen im Berufsleben,2,Gleichstellung der Geschlechter in geschäftlic...,0.654030
202,10,Damen im Berufsleben,3,über berufliche Tätigkeiten berichten,0.653680
220,11,Online-Bildung und Lehrgangseinschätzung,1,Online-Schulungen anbieten,0.842787


,docno,raw_text
615,3427,Was ist die Hauptstadt von Australien? Sydney ...
618,3430,Was ist die Hauptstadt von Japan? Peking Tokio...
628,3440,Was ist die Hauptstadt von Kanada? Toronto Mon...
650,3462,Was ist die Hauptstadt von Russland? Sankt Pet...
674,3486,Was ist der Unterschied zwischen AC und DC? AC...
1474,4286,Was ist der Unterschied zwischen Monopol und O...
2111,4923,Was ist der Unterschied zwischen Monopol und O...
2259,5071,Was ist ein gutes Passwort? Ein kurzes Passwor...
2392,5204,Welche Rolle spielt die Executive Summary in e...
2654,5466,Was ist ein Konsul? Ein Konsul ist ein politis...


,qid,raw_query,skill_label,sim,token_overlap_n,overlap_tokens
200,10,Damen im Berufsleben,die Gleichstellung der Geschlechter am Arbeits...,0.676048,0,
160,8,Staatskunst und internationale Steuerung,auswärtige Angelegenheiten,0.741712,0,
260,13,Laufbahnplanung und Eigenanalyse,Bestandsplanung durchführen,0.755688,0,
100,5,Erkenntnisvermögen und Eigenanalyse,Zustandserkennung,0.778430,0,
80,4,"Unternehmerische Fähigkeiten (Strategie, Ablau...",Rechte des geistigen Eigentums verwalten,0.798366,0,
280,14,Territoriale Grünplanung,Gebietsplanung anwenden,0.806194,0,
240,12,Speicher und Transportwesen,Betrieb von Transportanlagen,0.814161,0,
140,7,Darstellungs- und Vortragsmethoden,Grundierung auftragen,0.850355,0,
120,6,Informationsaufbereitung,Informationen ordnen,0.899655,0,
0,0,Digitaler Schutz,digitale Sicherheitsmaßnahmen anwenden,0.904953,0,
